In [2]:
# Menyiapkan package & Inisialisasi SparkSession

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import os
import shutil
import time
import pandas as pd
from datetime import datetime

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas10-StructuredStreaming-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:",spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [3]:
# A. Menyiapkan Streaming Source

# Menyiapkan folder simulasi streaming
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

# Mendefinisikan skema eksplisit
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

# Mendefinisikan Input Source
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming DataFrame berhasil didefinisikan. df_stream.isStreaming:", df_stream.isStreaming)

Streaming DataFrame berhasil didefinisikan. df_stream.isStreaming: True


In [4]:
# B. Agregasi per Cabang

# Tambahkan kolom turunan total_harga
df_stream_dengan_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi per cabang
agregasi_cabang = df_stream_dengan_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan Streaming Query ke memory sink
query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query agregasi_cabang dimulai! Status:", query_cabang.status)

Streaming query agregasi_cabang dimulai! Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


In [5]:
 # C. Agregasi Berbasis Window Waktu

# Membaca ulang streaming source dari folder yang sama
df_stream2 = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

df_stream2_dengan_total = df_stream2.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi window 2 menit berdasarkan waktu_pesan
agregasi_waktu = df_stream2_dengan_total.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

# Menjalankan Streaming Query kedua ke memory sink
query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query agregasi_waktu dimulai! Status:", query_waktu.status)

Streaming query agregasi_waktu dimulai! Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [6]:
# D. Menguji dengan Data Simulasi

waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# --- Batch 1 ---
pd.DataFrame({
    "pesanan_id": ["P1", "P2"],
    "nama_cabang": ["Magelang", "Yogyakarta"],
    "nama_menu": ["Kopi Susu", "Americano"],
    "qty": [2, 1],
    "harga_satuan": [18000, 15000],
    "waktu_pesan": [waktu_sekarang, waktu_sekarang],
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# --- Batch 2 ---
pd.DataFrame({
    "pesanan_id": ["P3", "P4"],
    "nama_cabang": ["Magelang", "Solo"],
    "nama_menu": ["Kopi Latte", "Espresso"],
    "qty": [1, 2],
    "harga_satuan": [22000, 12000],
    "waktu_pesan": [waktu_sekarang, waktu_sekarang],
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# --- Batch 3 ---
pd.DataFrame({
    "pesanan_id": ["P5"],
    "nama_cabang": ["Yogyakarta"],
    "nama_menu": ["Matcha Latte"],
    "qty": [2],
    "harga_satuan": [25000],
    "waktu_pesan": [waktu_sekarang],
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("Batch 3 dikirim. Menunggu Spark memproses...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

Batch 1 dikirim. Menunggu Spark memproses...


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|           36000|             1|
| Yogyakarta|           15000|             1|
+-----------+----------------+--------------+

Batch 2 dikirim. Menunggu Spark memproses...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|           58000|             2|
|       Solo|           24000|             1|
| Yogyakarta|           15000|             1|
+-----------+----------------+--------------+

Batch 3 dikirim. Menunggu Spark memproses...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
| Yogyakarta|           65000|             2|
|   Magelang|           58000|             2|
|       Solo|           24000|             1|
+-----------+----------------+----

In [7]:
# E. Sink Berbasis Berkas (Tantangan)

query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming query ke Parquet dimulai! Status:", query3.status)

waktu_sekarang = datetime.now()
print("Mengirim Batch 4...")
pd.DataFrame({
    "pesanan_id": ["P07", "P08"],
    "nama_cabang": ["Cabang Magelang", "Cabang Jogja"],
    "nama_menu": ["Roti Bakar", "Kopi Hitam"],
    "qty": [1, 2],
    "harga_satuan": [15000, 10000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv("tugas10_stream/batch_tugas_4.csv", index=False)

time.sleep(10)

print("Isi direktori tugas10_output:")
!ls tugas10_output

print("\nMembaca kembali data dari Parquet:")
df_parquet = spark.read.parquet("tugas10_output")
df_parquet.show()

Streaming query ke Parquet dimulai! Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
Mengirim Batch 4...
Isi direktori tugas10_output:
part-00000-e225cfa9-f5a3-48ca-9469-2e7208d5eb54-c000.snappy.parquet
part-00001-27614dea-c9b8-421e-aee0-ba7a86265f50-c000.snappy.parquet
part-00002-07c7bbbc-fba1-4926-81e2-ff873a5d7dc4-c000.snappy.parquet
part-00003-5b4599a8-00e2-4da5-ac86-610521257f1b-c000.snappy.parquet
_spark_metadata

Membaca kembali data dari Parquet:


+----------+---------------+------------+---+------------+-------------------+
|pesanan_id|    nama_cabang|   nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+---------------+------------+---+------------+-------------------+
|       P07|Cabang Magelang|  Roti Bakar|  1|       15000|2026-11-09 14:58:13|
|       P08|   Cabang Jogja|  Kopi Hitam|  2|       10000|2026-11-09 14:58:13|
|        P5|     Yogyakarta|Matcha Latte|  2|       25000|2026-11-09 14:57:53|
|        P1|       Magelang|   Kopi Susu|  2|       18000|2026-11-09 14:57:53|
|        P2|     Yogyakarta|   Americano|  1|       15000|2026-11-09 14:57:53|
|        P3|       Magelang|  Kopi Latte|  1|       22000|2026-11-09 14:57:53|
|        P4|           Solo|    Espresso|  2|       12000|2026-11-09 14:57:53|
+----------+---------------+------------+---+------------+-------------------+

